<a href="https://www.kaggle.com/code/shahporanpriyom/final-notebook?scriptVersionId=303192835" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import random
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
IMG_DIRS=['/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1',
          '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2'
         ]
OUT_DIR = "output"
splits = ["train", "val", "test"]
classes = ["cancer", "non_cancer"]
# Create directories
for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(OUT_DIR, split, cls), exist_ok=True)

# Load metadata
df = pd.read_csv('/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv')
# Binary label mapping
cancer_classes = ["mel", "bcc", "akiec"]
df["label"] = df["dx"].apply(
    lambda x: "cancer" if x in cancer_classes else "non_cancer"
)


In [ ]:
# Find image paths
def find_image(image_id):
    for d in IMG_DIRS:
        path = os.path.join(d, image_id + ".jpg")
        if os.path.exists(path):
            return path
    return None

df["image_path"] = df["image_id"].apply(find_image)
df = df.dropna(subset=["image_path"])


In [ ]:
# Train / val / test split (70 / 15 / 15)
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=18
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=18
)


In [ ]:
# Function to copy images
def copy_images(dataframe, split_name):
    for _, row in dataframe.iterrows():
        dst = os.path.join(
            OUT_DIR,
            split_name,
            row["label"],
            os.path.basename(row["image_path"])
        )
        shutil.copy(row["image_path"], dst)

copy_images(train_df, "train")
copy_images(val_df, "val")
copy_images(test_df, "test")

print("image merging and splitting completed!!!")

In [ ]:
for split in splits:
    cancer_path = os.path.join(OUT_DIR, split, "cancer")
    non_cancer_path = os.path.join(OUT_DIR, split, "non_cancer")
    
    cancer_files = os.listdir(cancer_path)
    non_cancer_files = os.listdir(non_cancer_path)
    
    cancer_count = len(cancer_files)
    non_cancer_count = len(non_cancer_files)
    
    if non_cancer_count > cancer_count:
        # Randomly select cancer_count non_cancer images to keep
        to_keep = random.sample(non_cancer_files, cancer_count)
        # Remove the excess
        for file in non_cancer_files:
            if file not in to_keep:
                os.remove(os.path.join(non_cancer_path, file))

# Print updated counts
    print(f"{split} cancer: {len(os.listdir(cancer_path))}")
    print(f"{split} non_cancer: {len(os.listdir(non_cancer_path))}")

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

TRAIN_DIR = "output/train"
classes = ["cancer", "non_cancer"]
images_per_class = 5
plt.figure(figsize=(10, 5))
img_index = 1
for cls in classes:
    cls_path = os.path.join(TRAIN_DIR, cls)
    imgs = random.sample(os.listdir(cls_path), min(images_per_class, len(os.listdir(cls_path))))
    for img_name in imgs:
        img_path = os.path.join(cls_path, img_name)
        img = Image.open(img_path)
        plt.subplot(2, images_per_class, img_index)
        plt.imshow(img)
        plt.title(cls)
        plt.axis("off")
        img_index += 1
plt.tight_layout()
plt.show()

In [ ]:
BASE_DIR = "output"
splits = ["train", "val", "test"]
classes = ["cancer", "non_cancer"]
counts = []
for split in splits:
    total = 0
    for cls in classes:
        path = os.path.join(BASE_DIR, split, cls)
        total += len(os.listdir(path))
    counts.append(total)

# Plot bar graph
plt.figure(figsize=(6,4))
plt.bar(splits, counts)
plt.title("Number of Images in Train / Val / Test (Balanced)")
plt.xlabel("Dataset Split")
plt.ylabel("Number of Images")
for i, v in enumerate(counts):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.show()

In [ ]:
TRAIN_DIR = "output/train"
classes = ["cancer", "non_cancer"]
counts = []
for cls in classes:
    cls_path = os.path.join(TRAIN_DIR, cls)
    counts.append(len(os.listdir(cls_path)))

# Plot bar graph
plt.figure(figsize=(5,4))
plt.bar(classes, counts)
plt.title("Training Data Class Distribution (Balanced)")
plt.xlabel("Class")
plt.ylabel("Number of Images")
for i, v in enumerate(counts):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.metrics import Precision, Recall, AUC
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 18

tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True
)

val_test_gen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_data = train_gen.flow_from_directory(
    "output/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_data = val_test_gen.flow_from_directory(
    "output/val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

test_data = val_test_gen.flow_from_directory(
    "output/test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

In [ ]:
model = keras.models.Sequential()

model.add(keras.layers.Input(shape=(128, 128, 3)))

model.add(keras.layers.Conv2D(32, 3, activation='relu', padding='same', kernel_initializer='he_normal'))
model.add(keras.layers.MaxPooling2D())
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Conv2D(64, 3, activation='relu', padding='same', kernel_initializer='he_normal'))
model.add(keras.layers.Conv2D(64, 3, activation='relu', padding='same', kernel_initializer='he_normal'))
model.add(keras.layers.MaxPooling2D())
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Conv2D(128, 3, activation='relu', padding='same', kernel_initializer='he_normal'))
model.add(keras.layers.Conv2D(128, 3, activation='relu', padding='same', kernel_initializer='he_normal'))
model.add(keras.layers.MaxPooling2D())
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Conv2D(256, 3, activation='relu', padding='same', kernel_initializer='he_normal'))
model.add(keras.layers.Conv2D(256, 3, activation='relu', padding='same', kernel_initializer='he_normal'))
model.add(keras.layers.MaxPooling2D())

model.add(keras.layers.Flatten())

model.add(keras.layers.Dropout(0.3))
model.add(keras.layers.Dense(256, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(128, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(64, activation='relu', kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())

model.add(keras.layers.Dense(
    1,
    activation='sigmoid',
    name='classifier'
))

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adamax(learning_rate=0.005),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        Precision(name='precision'),
        Recall(name='recall'),
        
    ]
)

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3
    )
]

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks
)

In [ ]:
y_prob = model.predict(test_data)
y_true = test_data.classes

In [ ]:
y_pred = (y_prob > 0.5).astype(int).ravel()

In [ ]:
print(classification_report(
    y_true,
    y_pred,
    target_names=['non_cancer', 'cancer']
))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
print(cm)

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

In [ ]:
y_pred = (y_prob > 0.6).astype(int)
print(classification_report(y_true, y_pred))


In [ ]:
print(confusion_matrix(y_true, y_pred))

In [ ]:
model.save('cnn_model.h5')



In [ ]:
import pandas as pd

meta_df = pd.read_csv(
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"
)

cancer_classes = ["mel", "bcc", "akiec"]

meta_df["binary_label"] = meta_df["dx"].apply(
    lambda x: 1 if x in cancer_classes else 0
)

In [ ]:
import os

CNN_BASE = "/kaggle/working/output"

def extract_image_ids(base_dir):
    image_ids = []

    for split in ["train", "val", "test"]:
        for cls in ["cancer", "non_cancer"]:
            folder = os.path.join(base_dir, split, cls)
            if not os.path.exists(folder):
                print("Missing:", folder)
                continue

            for fname in os.listdir(folder):
                if fname.lower().endswith(".jpg"):
                    image_ids.append(os.path.splitext(fname)[0])

    return set(image_ids)

cnn_image_ids = extract_image_ids(CNN_BASE)

print("Images used by CNN:", len(cnn_image_ids))

In [ ]:
meta_cnn = meta_df[meta_df["image_id"].isin(cnn_image_ids)].copy()

print(meta_cnn.shape)

In [ ]:
features = ["age", "sex", "localization"]
target = "binary_label"

X = meta_cnn[features]
y = meta_cnn[target]

In [ ]:
meta_cnn.tail()

In [ ]:
X["age"] = X["age"].fillna(X["age"].median())
X["sex"] = X["sex"].fillna("unknown")
X["localization"] = X["localization"].fillna("unknown")

In [ ]:
meta_cnn['localization'].value_counts()

In [ ]:
meta_cnn['sex'].value_counts()

In [ ]:
meta_cnn['age'].value_counts()

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pickle

encoders = {}

for col in ["sex", "localization"]:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

# Save encoders
with open("meta_label_encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    xgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )

    y_va_prob = xgb_model.predict_proba(X_va)[:, 1]
    auc = roc_auc_score(y_va, y_va_prob)
    cv_auc.append(auc)

    print(f"Fold {fold} AUC: {auc:.4f}")

print(f"\nMean CV AUC: {np.mean(cv_auc):.4f} ± {np.std(cv_auc):.4f}")

In [ ]:
xgb_model.save_model("xgboost_metadata_model.json")

In [ ]:
import numpy as np
import tensorflow as tf
from xgboost import XGBClassifier
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, precision_score, recall_score

# --- 1. SETUP & LOAD CNN ---
cnn_model = tf.keras.models.load_model('cnn_model.h5')
# Extract features from the BatchNormalization layer after the 64-unit Dense layer
feature_extractor = tf.keras.Model(inputs=cnn_model.inputs, 
                                   outputs=cnn_model.layers[-2].output)

# --- 2. GENERATORS (Match your file structure) ---
datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_directory(
    '/kaggle/working/output/train',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

test_generator = datagen.flow_from_directory(
    '/kaggle/working/output/test',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

# --- 3. FEATURE EXTRACTION ---
def extract_features(generator):
    features = feature_extractor.predict(generator)
    labels = generator.classes
    return features, labels

print("Extracting features (this may take a minute)...")
X_train_hyb, y_train_hyb = extract_features(train_generator)
X_test_hyb, y_test_hyb = extract_features(test_generator)

# --- 4. TRAIN NEW HYBRID XGBOOST ---
# We create a new model because the old one only expected 3 inputs
hybrid_xgb = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1)
hybrid_xgb.fit(X_train_hyb, y_train_hyb)

# --- 5. PREDICT & RESULTS ---
y_pred = hybrid_xgb.predict(X_test_hyb)

print(f"\n--- Updated Hybrid Performance (64 Features) ---")
print(f"Accuracy:  {accuracy_score(y_test_hyb, y_pred):.4f}")
print(f"Precision: {precision_score(y_test_hyb, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test_hyb, y_pred):.4f}")

# Save the new model
hybrid_xgb.save_model('hybrid_xgboost_64.json')


In [ ]:
import numpy as np
import tensorflow as tf
from xgboost import XGBClassifier
import pandas as pd
import pickle

# 1. Load Models and Encoders
cnn_model = tf.keras.models.load_model('cnn_model.h5')
xgb_model = XGBClassifier()
xgb_model.load_model('xgboost_metadata_model.json')

with open('meta_label_encoders.pkl', 'rb') as f:
    label_encoders = pickle.load(f)

def hybrid_inference(img_path, age, sex, localization):
    # --- CNN PATHWAY ---
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(128, 128))
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    cnn_prob = cnn_model.predict(img_array, verbose=0)[0][0]

    # --- XGBOOST PATHWAY ---
    # Encode metadata (assuming same order as training: age, sex, localization)
    sex_enc = label_encoders['sex'].transform([sex])[0]
    loc_enc = label_encoders['localization'].transform([localization])[0]
    meta_features = np.array([[age, sex_enc, loc_enc]])
    
    # Get XGB probability for class 1 (cancer)
    xgb_prob = xgb_model.predict_proba(meta_features)[0][1]

    
    # Formula: 0.6 * CNN + 0.4 * XGB
    hybrid_prob = (0.6 * cnn_prob) + (0.4 * xgb_prob)
    prediction = 1 if hybrid_prob >= 0.4 else 0
    
    return "CANCER" if prediction == 1 else "NON_CANCER", hybrid_prob



In [ ]:
# Example Usage
# 
label, score = hybrid_inference(
    img_path='/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1/ISIC_0024315.jpg',
    age=80, 
    sex='male', 
    localization='back'
)

print(f"Hybrid Prediction: {label} (Confidence Score: {score:.4f})")


In [ ]:
def get_gradcam_heatmap(img_array, model):
    img_tensor = tf.cast(np.expand_dims(img_array, axis=0), tf.float32)
    
    # 1. Identify the last Conv2D layer
    last_conv_layer_name = [l.name for l in model.layers if "conv2d" in l.name][-1]
    
    # 2. Create a sub-model for the feature extraction part
    # We find the index of the last conv layer to split the model
    conv_layer_idx = [i for i, l in enumerate(model.layers) if l.name == last_conv_layer_name][0]
    
    # Model 1: Input to Last Conv Layer
    layer_before_conv = tf.keras.Model(inputs=model.inputs, outputs=model.layers[conv_layer_idx].output)
    
    # Model 2: From Last Conv Layer to Final Prediction
    # We define this manually inside the tape to ensure gradients flow
    remaining_layers = model.layers[conv_layer_idx + 1:]

    with tf.GradientTape() as tape:
        # Forward pass to the convolutional layer
        last_conv_output = layer_before_conv(img_tensor)
        tape.watch(last_conv_output)
        
        # Forward pass through the rest of the model
        x = last_conv_output
        for layer in remaining_layers:
            x = layer(x)
        preds = x
        
        # Target the prediction score (class 0)
        class_channel = preds[:, 0]

    # 3. Calculate Gradients (now guaranteed to not be None)
    grads = tape.gradient(class_channel, last_conv_output)
    
    # 4. Global average pooling and Weighted Sum
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_output = last_conv_output[0]
    heatmap = last_conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # 5. Normalize
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2
import numpy as np

def display_gradcam(img, heatmap, alpha=0.4):
    # 1. Rescale heatmap to a range 0-255
    heatmap = np.uint8(255 * heatmap)

    # 2. Use jet colormap to colorize heatmap
    jet = cm.get_cmap("jet")

    # 3. Use RGB values of the colormap
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]

    # 4. Create an image with RGB colorized heatmap
    jet_heatmap = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
    jet_heatmap = tf.keras.preprocessing.image.img_to_array(jet_heatmap)

    # 5. Superimpose the heatmap on original image
    superimposed_img = jet_heatmap * alpha + (img * 255)
    superimposed_img = tf.keras.preprocessing.image.array_to_img(superimposed_img)

    # 6. Displaying the results
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(img)
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.title("Grad-CAM")
    plt.imshow(superimposed_img)
    plt.axis("off")
    plt.show()


In [ ]:
# 1. Prepare image
img_path = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1/ISIC_0024324.jpg'
img = tf.keras.preprocessing.image.load_img(img_path, target_size=(128, 128))
img_arr = tf.keras.preprocessing.image.img_to_array(img) / 255.0

# 2. Generate Heatmap
heatmap = get_gradcam_heatmap(img_arr, cnn_model)

# 3. Display Result
display_gradcam(img_arr, heatmap)
